# Project A: Market-Level Sentiment Trading Strategy

## Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Data Loading and Initial Setup

In [ ]:
# Load datasets
labeled_df = pd.read_csv('../tut/Tutorial 3-20260304/wsj_finbert_labeled_16000.csv')
headlines_df = pd.read_csv('./ECOM217_data/Shared_wsj_headlines_2023.csv')
spx_df = pd.read_csv('./ECOM217_data/ProjectA_spx_index_daily_2023.csv')

print(f"Labeled data shape: {labeled_df.shape}")
print(f"Full headlines shape: {headlines_df.shape}")
print(f"SPX data shape: {spx_df.shape}")

## Part A: NLP Pipeline

### Benchmark Model: TF-IDF -> PCA -> Logistic Regression
Train model on the labeled subset and report metrics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Ensure binary/multi-class setup matches the requirement. Let's see sentiment labels:
# Here we will drop 'neutral' if we only want positive/negative, or map it if it's multiclass.
labeled_df = labeled_df.dropna(subset=['sentiment', 'headline'])
X = labeled_df['headline']
y = labeled_df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 1. TF-IDF
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 2. PCA (TruncatedSVD for sparse matrices)
n_components = min(100, X_train_tfidf.shape[1] - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_pca = svd.fit_transform(X_train_tfidf)
X_test_pca = svd.transform(X_test)

# 3. Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_pca, y_train)
y_pred = lr.predict(X_test_pca)

print("Benchmark Model Evaluation:")
print(classification_report(y_test, y_pred))

### Apply to Full Dataset (~146K headlines)

In [ ]:
# Predict on the full headlines dataset
headlines_df['headline'] = headlines_df['headline'].fillna('')
X_full_tfidf = tfidf.transform(headlines_df['headline'])
X_full_pca = svd.transform(X_full_tfidf)

headlines_df['sentiment_pred'] = lr.predict(X_full_pca)
display(headlines_df['sentiment_pred'].value_counts())

### Advanced Model: LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

# Convert targets to binary (assuming binary labels)
train_texts = X_train.values.astype(str)
test_texts = X_test.values.astype(str)
y_train_num = (y_train == 'positive').astype(int).values
y_test_num = (y_test == 'positive').astype(int).values

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_texts)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=50)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=50)

model = Sequential([
    Embedding(5000, 64, input_length=50),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train_seq, y_train_num, epochs=3, validation_data=(X_test_seq, y_test_num), batch_size=64, verbose=1)

y_pred_prob_lstm = model.predict(X_test_seq)
y_pred_lstm = (y_pred_prob_lstm > 0.5).astype(int)


### 7 Required Visualizations

In [ ]:
# 1. Accuracy bar chart
# Re-calculate LogReg accuracy locally for plotting
y_pred_num = (y_pred == 'positive').astype(int)
acc_logreg = accuracy_score((y_test == 'positive').astype(int), y_pred_num)
acc_lstm = accuracy_score(y_test_num, y_pred_lstm)

plt.figure(figsize=(6,4))
sns.barplot(x=['LogReg (TF-IDF+PCA)', 'LSTM'], y=[acc_logreg, acc_lstm])
plt.title('Accuracy Comparison')
plt.ylabel('Accuracy')
plt.show()


In [ ]:
# 2. Top TF-IDF terms: positive vs. negative
# We use LogisticRegression coefs to find top terms
feature_names = tfidf.get_feature_names_out()
# Retrain LogReg directly on TF-IDF for interpretable coefficients
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

coefs = lr_tfidf.coef_[0]
top_positive_idx = np.argsort(coefs)[-10:]
top_negative_idx = np.argsort(coefs)[:10]

top_pos_words = [feature_names[i] for i in top_positive_idx]
top_neg_words = [feature_names[i] for i in top_negative_idx]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.barh(top_pos_words, coefs[top_positive_idx], color='g')
plt.title('Top Positive Words')
plt.subplot(1,2,2)
plt.barh(top_neg_words, coefs[top_negative_idx], color='r')
plt.title('Top Negative Words')
plt.tight_layout()
plt.show()


In [ ]:
# 3. PCA 2D scatter by sentiment
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=y_train, alpha=0.5)
plt.title('PCA 2D Scatter of Headlines')
plt.show()


In [ ]:
# 4. K-Means clustering with representative headlines
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X_train_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=clusters, palette='Set1', alpha=0.5)
plt.title('K-Means Clusters (k=2)')
plt.show()

# Show representative headlines
for i in range(2):
    center = kmeans.cluster_centers_[i]
    distances = np.linalg.norm(X_train_pca - center, axis=1)
    closest_idx = np.argmin(distances)
    print(f"Cluster {i} rep: {X_train.iloc[closest_idx]}")


In [ ]:
# 5. Precision / Recall / F1 table
metrics_df = pd.DataFrame(
    precision_recall_fscore_support(y_test, y_pred, average=None, labels=['negative', 'positive']),
    index=['Precision', 'Recall', 'F1', 'Support'],
    columns=['Negative', 'Positive']
).T
display(metrics_df)


In [ ]:
# 6. Most-confident correct + incorrect predictions
probs = lr.predict_proba(X_test_pca)
confidences = np.max(probs, axis=1)

results_df = pd.DataFrame({
    'text': X_test,
    'true': y_test,
    'pred': y_pred,
    'confidence': confidences
})

correct = results_df[results_df['true'] == results_df['pred']]
incorrect = results_df[results_df['true'] != results_df['pred']]

print("Most confident CORRECT:")
display(correct.sort_values('confidence', ascending=False).head(3))

print("Most confident INCORRECT:")
display(incorrect.sort_values('confidence', ascending=False).head(3))


In [ ]:
# 7. Training curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('LSTM Loss Curve')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('LSTM Accuracy Curve')
plt.legend()
plt.tight_layout()
plt.show()


### Advanced Model: LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

# Convert targets to binary (assuming binary labels)
train_texts = X_train.values.astype(str)
test_texts = X_test.values.astype(str)
y_train_num = (y_train == 'positive').astype(int).values
y_test_num = (y_test == 'positive').astype(int).values

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_texts)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(train_texts), maxlen=50)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(test_texts), maxlen=50)

model = Sequential([
    Embedding(5000, 64, input_length=50),
    LSTM(32),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train_seq, y_train_num, epochs=3, validation_data=(X_test_seq, y_test_num), batch_size=64, verbose=1)

y_pred_prob_lstm = model.predict(X_test_seq)
y_pred_lstm = (y_pred_prob_lstm > 0.5).astype(int)


### 7 Required Visualizations

In [ ]:
# 1. Accuracy bar chart
# Re-calculate LogReg accuracy locally for plotting
y_pred_num = (y_pred == 'positive').astype(int)
acc_logreg = accuracy_score((y_test == 'positive').astype(int), y_pred_num)
acc_lstm = accuracy_score(y_test_num, y_pred_lstm)

plt.figure(figsize=(6,4))
sns.barplot(x=['LogReg (TF-IDF+PCA)', 'LSTM'], y=[acc_logreg, acc_lstm])
plt.title('Accuracy Comparison')
plt.ylabel('Accuracy')
plt.show()


In [ ]:
# 2. Top TF-IDF terms: positive vs. negative
# We use LogisticRegression coefs to find top terms
feature_names = tfidf.get_feature_names_out()
# Retrain LogReg directly on TF-IDF for interpretable coefficients
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)

coefs = lr_tfidf.coef_[0]
top_positive_idx = np.argsort(coefs)[-10:]
top_negative_idx = np.argsort(coefs)[:10]

top_pos_words = [feature_names[i] for i in top_positive_idx]
top_neg_words = [feature_names[i] for i in top_negative_idx]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.barh(top_pos_words, coefs[top_positive_idx], color='g')
plt.title('Top Positive Words')
plt.subplot(1,2,2)
plt.barh(top_neg_words, coefs[top_negative_idx], color='r')
plt.title('Top Negative Words')
plt.tight_layout()
plt.show()


In [ ]:
# 3. PCA 2D scatter by sentiment
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=y_train, alpha=0.5)
plt.title('PCA 2D Scatter of Headlines')
plt.show()


In [ ]:
# 4. K-Means clustering with representative headlines
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X_train_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=clusters, palette='Set1', alpha=0.5)
plt.title('K-Means Clusters (k=2)')
plt.show()

# Show representative headlines
for i in range(2):
    center = kmeans.cluster_centers_[i]
    distances = np.linalg.norm(X_train_pca - center, axis=1)
    closest_idx = np.argmin(distances)
    print(f"Cluster {i} rep: {X_train.iloc[closest_idx]}")


In [ ]:
# 5. Precision / Recall / F1 table
metrics_df = pd.DataFrame(
    precision_recall_fscore_support(y_test, y_pred, average=None, labels=['negative', 'positive']),
    index=['Precision', 'Recall', 'F1', 'Support'],
    columns=['Negative', 'Positive']
).T
display(metrics_df)


In [ ]:
# 6. Most-confident correct + incorrect predictions
probs = lr.predict_proba(X_test_pca)
confidences = np.max(probs, axis=1)

results_df = pd.DataFrame({
    'text': X_test,
    'true': y_test,
    'pred': y_pred,
    'confidence': confidences
})

correct = results_df[results_df['true'] == results_df['pred']]
incorrect = results_df[results_df['true'] != results_df['pred']]

print("Most confident CORRECT:")
display(correct.sort_values('confidence', ascending=False).head(3))

print("Most confident INCORRECT:")
display(incorrect.sort_values('confidence', ascending=False).head(3))


In [ ]:
# 7. Training curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('LSTM Loss Curve')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('LSTM Accuracy Curve')
plt.legend()
plt.tight_layout()
plt.show()


# Part B: Strategy Design

### Step 1: Build Your Signal & Step 2: Financial Intuition

In [ ]:
# Part B: Strategy Design

# 1. Base Signal mapping
headlines_df['date'] = pd.to_datetime(headlines_df['date'])

# For binary sentiment (1=positive, 0=negative), let's calculate N_pos and N_neg
daily_counts = headlines_df.groupby('date')['sentiment_pred'].agg(
    N_pos=lambda x: (x == 1).sum(),
    N_neg=lambda x: (x == 0).sum(),
    N_total='count'
).reset_index()

daily_counts['S_t'] = (daily_counts['N_pos'] - daily_counts['N_neg']) / daily_counts['N_total']

# Merge with S&P500 Returns
spx_df['date'] = pd.to_datetime(spx_df['date'])
# Create a next-day SPX return for trading
# The strategy: Use sentiment from day t to invest on day t+1
spx_df['next_day_ret'] = spx_df['sprtrn'].shift(-1)

merged_df = pd.merge(spx_df, daily_counts, on='date', how='left')
merged_df['S_t'] = merged_df['S_t'].fillna(0) # Assume 0 sentiment if no news

# Filter to test period matching the requirements (2022 to 2023)
test_df = merged_df[(merged_df['date'] >= '2022-01-01') & (merged_df['date'] <= '2023-12-31')].copy()

# Step 2: Financial intuition
# 1. Base / Momentum strategy (S_t > 0 -> buy)
test_df['w_momentum'] = (test_df['S_t'] > 0).astype(int)
test_df['ret_momentum'] = test_df['w_momentum'] * test_df['next_day_ret']

# 2. Mean-Reversion strategy (S_t < 0 -> buy)
test_df['w_reversion'] = (test_df['S_t'] < 0).astype(int)
test_df['ret_reversion'] = test_df['w_reversion'] * test_df['next_day_ret']

# 3. Surprise Strategy (Deviation from 30 day avg)
merged_df['S_t_30d_avg'] = merged_df['S_t'].rolling(window=30, min_periods=1).mean().shift(1)
test_df['S_t_30d_avg'] = merged_df.loc[test_df.index, 'S_t_30d_avg']
test_df['S_surprise'] = test_df['S_t'] - test_df['S_t_30d_avg']
test_df['w_surprise'] = (test_df['S_surprise'] > 0).astype(int)
test_df['ret_surprise'] = test_df['w_surprise'] * test_df['next_day_ret']

print("Evaluating Strategy Implementations (Annualised Returns in Test Period):")
print(f"Buy/Hold SPX:      {test_df['next_day_ret'].mean() * 252:.4f}")
print(f"Momentum Strategy: {test_df['ret_momentum'].mean() * 252:.4f}")
print(f"Mean-Reversion:    {test_df['ret_reversion'].mean() * 252:.4f}")
print(f"Surprise Strategy: {test_df['ret_surprise'].mean() * 252:.4f}")


### Step 3: Topic Analysis

In [ ]:
# Predict clusters for the full headline dataset
headlines_df['headline_clean'] = headlines_df['headline'].fillna('')
# We trained tfidf and svd earlier, reuse them:
X_full_pca = svd.transform(tfidf.transform(headlines_df['headline_clean']))
headlines_df['cluster'] = kmeans.predict(X_full_pca)

headlines_ret_merged = pd.merge(headlines_df, spx_df, on='date', how='left')
topic_returns = headlines_ret_merged.groupby('cluster')['next_day_ret'].mean()

print("Average SPX next-day return by news topic (cluster):")
print(topic_returns)

# Let's inspect some sample headlines from each topic to find the narrative
for c in topic_returns.index:
    print(f"\n--- Cluster {c} Headlines ---")
    sample_texts = headlines_df[headlines_df['cluster'] == c]['headline_clean'].sample(5, random_state=42)
    for text in sample_texts:
        print(f" - {text}")


### Step 4: Error Analysis

In [ ]:
# Find days with high positive sentiment but severe market drop (e.g., S_t > 0.5, ret < -2%)
error_days = test_df[(test_df['S_t'] > 0.5) & (test_df['next_day_ret'] < -0.02)]

print("Notable Error Days (Strong positive sentiment, but SPX plummeted next day):")
display(error_days[['date', 'S_t', 'next_day_ret']].head())

if not error_days.empty:
    bad_day = error_days.iloc[0]['date']
    print(f"\nInvestigating headlines published on {bad_day.date()}:")
    bad_day_news = headlines_df[headlines_df['date'] == bad_day]['headline_clean']
    for text in bad_day_news.head(10):
        print(f" - {text}")


# Part B: Strategy Design

### Step 1: Build Your Signal & Step 2: Financial Intuition

In [ ]:
# Part B: Strategy Design

# 1. Base Signal mapping
headlines_df['date'] = pd.to_datetime(headlines_df['date'])

# For binary sentiment (1=positive, 0=negative), let's calculate N_pos and N_neg
daily_counts = headlines_df.groupby('date')['sentiment_pred'].agg(
    N_pos=lambda x: (x == 1).sum(),
    N_neg=lambda x: (x == 0).sum(),
    N_total='count'
).reset_index()

daily_counts['S_t'] = (daily_counts['N_pos'] - daily_counts['N_neg']) / daily_counts['N_total']

# Merge with S&P500 Returns
spx_df['date'] = pd.to_datetime(spx_df['date'])
# Create a next-day SPX return for trading
# The strategy: Use sentiment from day t to invest on day t+1
spx_df['next_day_ret'] = spx_df['sprtrn'].shift(-1)

merged_df = pd.merge(spx_df, daily_counts, on='date', how='left')
merged_df['S_t'] = merged_df['S_t'].fillna(0) # Assume 0 sentiment if no news

# Filter to test period matching the requirements (2022 to 2023)
test_df = merged_df[(merged_df['date'] >= '2022-01-01') & (merged_df['date'] <= '2023-12-31')].copy()

# Step 2: Financial intuition
# 1. Base / Momentum strategy (S_t > 0 -> buy)
test_df['w_momentum'] = (test_df['S_t'] > 0).astype(int)
test_df['ret_momentum'] = test_df['w_momentum'] * test_df['next_day_ret']

# 2. Mean-Reversion strategy (S_t < 0 -> buy)
test_df['w_reversion'] = (test_df['S_t'] < 0).astype(int)
test_df['ret_reversion'] = test_df['w_reversion'] * test_df['next_day_ret']

# 3. Surprise Strategy (Deviation from 30 day avg)
merged_df['S_t_30d_avg'] = merged_df['S_t'].rolling(window=30, min_periods=1).mean().shift(1)
test_df['S_t_30d_avg'] = merged_df.loc[test_df.index, 'S_t_30d_avg']
test_df['S_surprise'] = test_df['S_t'] - test_df['S_t_30d_avg']
test_df['w_surprise'] = (test_df['S_surprise'] > 0).astype(int)
test_df['ret_surprise'] = test_df['w_surprise'] * test_df['next_day_ret']

print("Evaluating Strategy Implementations (Annualised Returns in Test Period):")
print(f"Buy/Hold SPX:      {test_df['next_day_ret'].mean() * 252:.4f}")
print(f"Momentum Strategy: {test_df['ret_momentum'].mean() * 252:.4f}")
print(f"Mean-Reversion:    {test_df['ret_reversion'].mean() * 252:.4f}")
print(f"Surprise Strategy: {test_df['ret_surprise'].mean() * 252:.4f}")


### Step 3: Topic Analysis

In [ ]:
# Predict clusters for the full headline dataset
headlines_df['headline_clean'] = headlines_df['headline'].fillna('')
# We trained tfidf and svd earlier, reuse them:
X_full_pca = svd.transform(tfidf.transform(headlines_df['headline_clean']))
headlines_df['cluster'] = kmeans.predict(X_full_pca)

headlines_ret_merged = pd.merge(headlines_df, spx_df, on='date', how='left')
topic_returns = headlines_ret_merged.groupby('cluster')['next_day_ret'].mean()

print("Average SPX next-day return by news topic (cluster):")
print(topic_returns)

# Let's inspect some sample headlines from each topic to find the narrative
for c in topic_returns.index:
    print(f"\n--- Cluster {c} Headlines ---")
    sample_texts = headlines_df[headlines_df['cluster'] == c]['headline_clean'].sample(5, random_state=42)
    for text in sample_texts:
        print(f" - {text}")


### Step 4: Error Analysis

In [ ]:
# Find days with high positive sentiment but severe market drop (e.g., S_t > 0.5, ret < -2%)
error_days = test_df[(test_df['S_t'] > 0.5) & (test_df['next_day_ret'] < -0.02)]

print("Notable Error Days (Strong positive sentiment, but SPX plummeted next day):")
display(error_days[['date', 'S_t', 'next_day_ret']].head())

if not error_days.empty:
    bad_day = error_days.iloc[0]['date']
    print(f"\nInvestigating headlines published on {bad_day.date()}:")
    bad_day_news = headlines_df[headlines_df['date'] == bad_day]['headline_clean']
    for text in bad_day_news.head(10):
        print(f" - {text}")


# Part B: Strategy Design

### Step 1: Build Your Signal & Step 2: Financial Intuition

In [ ]:
# Part B: Strategy Design

# 1. Base Signal mapping
headlines_df['date'] = pd.to_datetime(headlines_df['date'])

# For binary sentiment (1=positive, 0=negative), let's calculate N_pos and N_neg
daily_counts = headlines_df.groupby('date')['sentiment_pred'].agg(
    N_pos=lambda x: (x == 1).sum(),
    N_neg=lambda x: (x == 0).sum(),
    N_total='count'
).reset_index()

daily_counts['S_t'] = (daily_counts['N_pos'] - daily_counts['N_neg']) / daily_counts['N_total']

# Merge with S&P500 Returns
spx_df['date'] = pd.to_datetime(spx_df['date'])
# Create a next-day SPX return for trading
# The strategy: Use sentiment from day t to invest on day t+1
spx_df['next_day_ret'] = spx_df['sprtrn'].shift(-1)

merged_df = pd.merge(spx_df, daily_counts, on='date', how='left')
merged_df['S_t'] = merged_df['S_t'].fillna(0) # Assume 0 sentiment if no news

# Filter to test period matching the requirements (2022 to 2023)
test_df = merged_df[(merged_df['date'] >= '2022-01-01') & (merged_df['date'] <= '2023-12-31')].copy()

# Step 2: Financial intuition
# 1. Base / Momentum strategy (S_t > 0 -> buy)
test_df['w_momentum'] = (test_df['S_t'] > 0).astype(int)
test_df['ret_momentum'] = test_df['w_momentum'] * test_df['next_day_ret']

# 2. Mean-Reversion strategy (S_t < 0 -> buy)
test_df['w_reversion'] = (test_df['S_t'] < 0).astype(int)
test_df['ret_reversion'] = test_df['w_reversion'] * test_df['next_day_ret']

# 3. Surprise Strategy (Deviation from 30 day avg)
merged_df['S_t_30d_avg'] = merged_df['S_t'].rolling(window=30, min_periods=1).mean().shift(1)
test_df['S_t_30d_avg'] = merged_df.loc[test_df.index, 'S_t_30d_avg']
test_df['S_surprise'] = test_df['S_t'] - test_df['S_t_30d_avg']
test_df['w_surprise'] = (test_df['S_surprise'] > 0).astype(int)
test_df['ret_surprise'] = test_df['w_surprise'] * test_df['next_day_ret']

print("Evaluating Strategy Implementations (Annualised Returns in Test Period):")
print(f"Buy/Hold SPX:      {test_df['next_day_ret'].mean() * 252:.4f}")
print(f"Momentum Strategy: {test_df['ret_momentum'].mean() * 252:.4f}")
print(f"Mean-Reversion:    {test_df['ret_reversion'].mean() * 252:.4f}")
print(f"Surprise Strategy: {test_df['ret_surprise'].mean() * 252:.4f}")


### Step 3: Topic Analysis

In [ ]:
# Predict clusters for the full headline dataset
headlines_df['headline_clean'] = headlines_df['headline'].fillna('')
# We trained tfidf and svd earlier, reuse them:
X_full_pca = svd.transform(tfidf.transform(headlines_df['headline_clean']))
headlines_df['cluster'] = kmeans.predict(X_full_pca)

headlines_ret_merged = pd.merge(headlines_df, spx_df, on='date', how='left')
topic_returns = headlines_ret_merged.groupby('cluster')['next_day_ret'].mean()

print("Average SPX next-day return by news topic (cluster):")
print(topic_returns)

# Let's inspect some sample headlines from each topic to find the narrative
for c in topic_returns.index:
    print(f"\n--- Cluster {c} Headlines ---")
    sample_texts = headlines_df[headlines_df['cluster'] == c]['headline_clean'].sample(5, random_state=42)
    for text in sample_texts:
        print(f" - {text}")


### Step 4: Error Analysis

In [ ]:
# Find days with high positive sentiment but severe market drop (e.g., S_t > 0.5, ret < -2%)
error_days = test_df[(test_df['S_t'] > 0.5) & (test_df['next_day_ret'] < -0.02)]

print("Notable Error Days (Strong positive sentiment, but SPX plummeted next day):")
display(error_days[['date', 'S_t', 'next_day_ret']].head())

if not error_days.empty:
    bad_day = error_days.iloc[0]['date']
    print(f"\nInvestigating headlines published on {bad_day.date()}:")
    bad_day_news = headlines_df[headlines_df['date'] == bad_day]['headline_clean']
    for text in bad_day_news.head(10):
        print(f" - {text}")


# Part B: Strategy Design

### Step 1: Build Your Signal & Step 2: Financial Intuition

In [ ]:
# Part B: Strategy Design

# 1. Base Signal mapping
headlines_df['date'] = pd.to_datetime(headlines_df['date'])

# For binary sentiment (1=positive, 0=negative), let's calculate N_pos and N_neg
daily_counts = headlines_df.groupby('date')['sentiment_pred'].agg(
    N_pos=lambda x: (x == 1).sum(),
    N_neg=lambda x: (x == 0).sum(),
    N_total='count'
).reset_index()

daily_counts['S_t'] = (daily_counts['N_pos'] - daily_counts['N_neg']) / daily_counts['N_total']

# Merge with S&P500 Returns
spx_df['date'] = pd.to_datetime(spx_df['date'])
# Create a next-day SPX return for trading
# The strategy: Use sentiment from day t to invest on day t+1
spx_df['next_day_ret'] = spx_df['sprtrn'].shift(-1)

merged_df = pd.merge(spx_df, daily_counts, on='date', how='left')
merged_df['S_t'] = merged_df['S_t'].fillna(0) # Assume 0 sentiment if no news

# Filter to test period matching the requirements (2022 to 2023)
test_df = merged_df[(merged_df['date'] >= '2022-01-01') & (merged_df['date'] <= '2023-12-31')].copy()

# Step 2: Financial intuition
# 1. Base / Momentum strategy (S_t > 0 -> buy)
test_df['w_momentum'] = (test_df['S_t'] > 0).astype(int)
test_df['ret_momentum'] = test_df['w_momentum'] * test_df['next_day_ret']

# 2. Mean-Reversion strategy (S_t < 0 -> buy)
test_df['w_reversion'] = (test_df['S_t'] < 0).astype(int)
test_df['ret_reversion'] = test_df['w_reversion'] * test_df['next_day_ret']

# 3. Surprise Strategy (Deviation from 30 day avg)
merged_df['S_t_30d_avg'] = merged_df['S_t'].rolling(window=30, min_periods=1).mean().shift(1)
test_df['S_t_30d_avg'] = merged_df.loc[test_df.index, 'S_t_30d_avg']
test_df['S_surprise'] = test_df['S_t'] - test_df['S_t_30d_avg']
test_df['w_surprise'] = (test_df['S_surprise'] > 0).astype(int)
test_df['ret_surprise'] = test_df['w_surprise'] * test_df['next_day_ret']

print("Evaluating Strategy Implementations (Annualised Returns in Test Period):")
print(f"Buy/Hold SPX:      {test_df['next_day_ret'].mean() * 252:.4f}")
print(f"Momentum Strategy: {test_df['ret_momentum'].mean() * 252:.4f}")
print(f"Mean-Reversion:    {test_df['ret_reversion'].mean() * 252:.4f}")
print(f"Surprise Strategy: {test_df['ret_surprise'].mean() * 252:.4f}")


### Step 3: Topic Analysis

In [ ]:
# Predict clusters for the full headline dataset
headlines_df['headline_clean'] = headlines_df['headline'].fillna('')
# We trained tfidf and svd earlier, reuse them:
X_full_pca = svd.transform(tfidf.transform(headlines_df['headline_clean']))
headlines_df['cluster'] = kmeans.predict(X_full_pca)

headlines_ret_merged = pd.merge(headlines_df, spx_df, on='date', how='left')
topic_returns = headlines_ret_merged.groupby('cluster')['next_day_ret'].mean()

print("Average SPX next-day return by news topic (cluster):")
print(topic_returns)

# Let's inspect some sample headlines from each topic to find the narrative
for c in topic_returns.index:
    print(f"\n--- Cluster {c} Headlines ---")
    sample_texts = headlines_df[headlines_df['cluster'] == c]['headline_clean'].sample(5, random_state=42)
    for text in sample_texts:
        print(f" - {text}")


### Step 4: Error Analysis

In [ ]:
# Find days with high positive sentiment but severe market drop (e.g., S_t > 0.5, ret < -2%)
error_days = test_df[(test_df['S_t'] > 0.5) & (test_df['next_day_ret'] < -0.02)]

print("Notable Error Days (Strong positive sentiment, but SPX plummeted next day):")
display(error_days[['date', 'S_t', 'next_day_ret']].head())

if not error_days.empty:
    bad_day = error_days.iloc[0]['date']
    print(f"\nInvestigating headlines published on {bad_day.date()}:")
    bad_day_news = headlines_df[headlines_df['date'] == bad_day]['headline_clean']
    for text in bad_day_news.head(10):
        print(f" - {text}")


# Part C: Evaluation

In [ ]:
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

sns.set_style('whitegrid')

# First, merge FF3 factors to our dataset so we can calculate the alpha
ff_df = pd.read_csv('./ECOM217_data/Shared_ff_factors_daily_2023.csv')
ff_df['date'] = pd.to_datetime(ff_df['date'], format='%Y-%m-%d', errors='coerce')
# We must shift FF factors to align with next_day_ret, or just merge them against the trading day t+1
# Since our return is on day t+1, the actual risk-free rate and factors we care about are for day t+1.
ff_df['trading_day'] = ff_df['date']

# Create trading_day in merged_df which is date + 1 trading day (since date is publication date)
# For simplicity and since SPX next_day_ret already maps directly to t+1, let's create a date_t1 column
# spx_df dates are exact trading dates (t). So shift(-1) gets actual t+1 return.
# We need to map FF3 factors to t+1. 
temp_spx = spx_df[['date']].copy()
temp_spx['trading_day'] = temp_spx['date'].shift(-1)
merged_df = pd.merge(merged_df, temp_spx, on='date', how='left')

# Join FF3 on trading_day
merged_df = pd.merge(merged_df, ff_df, left_on='trading_day', right_on='trading_day', how='left', suffixes=('', '_ff'))

# Also make sure the strategy w_t and ret is applied to the full dataset for In-Sample vs Out-Of-Sample
merged_df['w_momentum'] = (merged_df['S_t'] > 0).astype(int)
merged_df['ret_momentum'] = merged_df['w_momentum'] * merged_df['next_day_ret']

# And drop NaNs for reliable regression
eval_df = merged_df.dropna(subset=['next_day_ret', 'mktrf', 'smb', 'hml', 'rf']).copy()

train_eval = eval_df[(eval_df['date'] >= '2016-01-01') & (eval_df['date'] <= '2021-12-31')].copy()
test_eval = eval_df[(eval_df['date'] >= '2022-01-01') & (eval_df['date'] <= '2023-12-31')].copy()

# Helper function to compute max drawdown
def calc_max_drawdown(returns):
    cum_returns = (1 + returns).cumprod()
    peak = cum_returns.cummax()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

def evaluate_strategy(df, strat_col='ret_momentum', weight_col='w_momentum'):
    # Tier 1
    cum_ret = (1 + df[strat_col]).prod() - 1
    ann_ret = df[strat_col].mean() * 252
    
    # Tier 2
    ann_vol = df[strat_col].std() * np.sqrt(252)
    sharpe = (ann_ret - (df['rf'].mean() * 252)) / ann_vol if ann_vol != 0 else 0
    
    # FF3 Alpha & t-stat
    # Y = R_strategy - R_f
    y = df[strat_col] - df['rf']
    X = df[['mktrf', 'smb', 'hml']]
    X = sm.add_constant(X)
    
    try:
        model = sm.OLS(y, X).fit()
        alpha = model.params['const'] * 252 # Annualised
        t_stat = model.tvalues['const']
    except:
        alpha, t_stat = 0, 0
    
    # Tier 3
    mdd = calc_max_drawdown(df[strat_col])
    calmar = ann_ret / abs(mdd) if mdd != 0 else 0
    
    # Tier 4
    turnover = df[weight_col].diff().abs().fillna(0)
    cost_5bps = turnover * 0.0005
    cost_10bps = turnover * 0.0010
    
    net_5_ret = df[strat_col] - cost_5bps
    net_10_ret = df[strat_col] - cost_10bps
    
    ann_net_5 = net_5_ret.mean() * 252
    sharpe_5 = (ann_net_5 - (df['rf'].mean() * 252)) / (net_5_ret.std() * np.sqrt(252))
    
    ann_net_10 = net_10_ret.mean() * 252
    sharpe_10 = (ann_net_10 - (df['rf'].mean() * 252)) / (net_10_ret.std() * np.sqrt(252))
    
    print(f"--- Strategy Performance ---")
    print(f"Cumulative Return: {cum_ret:.2%}")
    print(f"Annualised Return: {ann_ret:.2%}")
    print(f"Sharpe Ratio:      {sharpe:.3f}")
    print(f"FF3 Alpha (Ann):   {alpha:.2%} (t-stat: {t_stat:.2f})")
    print(f"Max Drawdown:      {mdd:.2%}")
    print(f"Calmar Ratio:      {calmar:.3f}")
    print(f"Net Sharpe (5bps): {sharpe_5:.3f}")
    print(f"Net Sharpe(10bps): {sharpe_10:.3f}\n")

print(">>> OUT-OF-SAMPLE EVALUATION (2022-2023) <<<")
evaluate_strategy(test_eval)

print(">>> IN-SAMPLE EVALUATION (2016-2021) <<<")
evaluate_strategy(train_eval)


### Plot 1: Sentiment Time Series Overlaid with SPX Returns

In [ ]:
# Re-aggregate sentiment to a monthly rolling average so the plot isn't a chaotic mess
test_eval['S_t_smooth'] = test_eval['S_t'].rolling(10).mean()
test_eval['SPX_Cum'] = (1 + test_eval['next_day_ret']).cumprod()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(test_eval['trading_day'], test_eval['SPX_Cum'], color='blue', label='SPX Cumulative')
ax1.set_ylabel('SPX Cumulative Return', color='blue')

ax2 = ax1.twinx()
ax2.plot(test_eval['trading_day'], test_eval['S_t_smooth'], color='orange', alpha=0.6, label='10-Day Avg Sentiment')
ax2.set_ylabel('Sentiment Signal', color='orange')

plt.title('Plot 1: Sentiment Time Series (10-Day Avg) vs SPX Cumulative Returns (Out-of-Sample)')
fig.tight_layout()
plt.show()


### Plot 2: Scatter: Sentiment vs Next-Day Return

In [ ]:
slope, intercept, r_value, p_value, std_err = linregress(test_eval['S_t'], test_eval['next_day_ret'])

plt.figure(figsize=(8, 6))
sns.regplot(x='S_t', y='next_day_ret', data=test_eval, scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
plt.title(f"Plot 2: Sentiment vs Next-Day Return\nR-squared: {r_value**2:.5f}")
plt.xlabel("Daily Sentiment (S_t)")
plt.ylabel("Next-Day SPX Return")
plt.show()


### Plot 3: Cumulative Return: Strategy vs Buy-and-Hold vs 50/50

In [ ]:
test_eval['Cum_Hold'] = (1 + test_eval['next_day_ret']).cumprod()
test_eval['Cum_Strat'] = (1 + test_eval['ret_momentum']).cumprod()

# 50/50 strategy matches 50% wait in cash (0 return, implicitly excluding RF to keep it simple) and 50% in SPX
test_eval['ret_5050'] = test_eval['next_day_ret'] * 0.5
test_eval['Cum_5050'] = (1 + test_eval['ret_5050']).cumprod()

plt.figure(figsize=(10, 5))
plt.plot(test_eval['trading_day'], test_eval['Cum_Hold'], label='Buy & Hold (SPX)', color='black', linestyle='--')
plt.plot(test_eval['trading_day'], test_eval['Cum_Strat'], label='Momentum Strategy', color='green', linewidth=2)
plt.plot(test_eval['trading_day'], test_eval['Cum_5050'], label='50/50 (SPX/Cash)', color='red', linestyle=':')

plt.title('Plot 3: Cumulative Return Comparison (Test Period: 2022-2023)')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()
